# LOT 43 — Multi-Fidelity Autonomous Teacher Protocol

Dry-run uniquement : aucun entraînement, aucun nouveau MCTS. Le notebook lit les artefacts Lot41B/Lot42 depuis Google Drive et produit le protocole `MULTI_FIDELITY_TEACHER_V1`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/GlennEriss/songo-fish.git'
REPO_DIR = Path('/content/songo-fish')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
assert Path('pyproject.toml').is_file(), f'pyproject.toml absent de {Path.cwd()}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,perf,dev]'], check=True)
os.environ['PYTHONPATH'] = 'packages:apps/trainer/scripts'
print('Projet installé depuis :', Path.cwd())

In [ ]:
from google.colab import drive
from google.colab import files
DRIVE_AVAILABLE = False
try:
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
except Exception as exc:
    print('Drive indisponible, fallback téléchargement manuel:', repr(exc))

DRIVE_ROOT = Path('/content/drive/MyDrive/songo-ai') if DRIVE_AVAILABLE else Path('/content')
LOT41B = DRIVE_ROOT / 'experiments/lot41b_deep_mcts_strategic_convergence'
LOT42 = DRIVE_ROOT / 'experiments/lot42_hard_position_65536'
OUTPUT = DRIVE_ROOT / 'experiments/lot43_multi_fidelity_teacher_protocol'
EXPORT = DRIVE_ROOT / 'exports/lot43_results.tar.gz'
OUTPUT.mkdir(parents=True, exist_ok=True)
EXPORT.parent.mkdir(parents=True, exist_ok=True)
print('LOT41B =', LOT41B)
print('LOT42  =', LOT42)
print('OUTPUT =', OUTPUT)

In [ ]:
required = [
    LOT41B / 'decision.json',
    LOT41B / 'hard_positions.json',
    LOT42 / 'decision.json',
    LOT42 / 'search_65536_results.json',
    LOT42 / 'comparison_32768_65536.json',
    LOT42 / 'ultra_hard_positions.json',
]
for p in required:
    print(p, 'OK' if p.exists() else 'MISSING')
assert all(p.exists() for p in required), 'Artefacts Lot41B/Lot42 manquants dans Drive'

In [ ]:
RUN = [sys.executable, 'apps/trainer/scripts/run_srn_lot43.py']
subprocess.run(RUN + ['--stage', 'analyze', '--lot41b', str(LOT41B), '--lot42', str(LOT42), '--output', str(OUTPUT)], check=True)

In [ ]:
subprocess.run(RUN + ['--stage', 'export', '--output', str(OUTPUT), '--bundle', str(EXPORT)], check=True)
print('decision.json:', OUTPUT / 'decision.json')
print('export:', EXPORT)
if not DRIVE_AVAILABLE:
    files.download(str(EXPORT))
    files.download(str(Path(str(EXPORT) + '.sha256')))